In [10]:
import pandas as pd
from pathlib import Path

# 数据文件真实名字是「奶茶店周报.csv」，在同级 ttttt/ 子目录里，不是「奶茶店周报数据.csv」。
# 更关键的是不能只写文件名：内核工作目录不一定是 notebook 所在目录（VS Code 常从工作区根启动），
# 换个启动方式就 FileNotFoundError，所以按「本级 → 各级父目录 → 子目录」逐级定位。
def 定位周报文件(文件名: str = "奶茶店周报.csv") -> Path:
    """返回第一个命中的 CSV 路径；找不到就抛出带搜索范围的 FileNotFoundError。"""
    cwd = Path.cwd()
    for 候选 in (cwd / 文件名, *(父目录 / 文件名 for 父目录 in cwd.parents)):
        if 候选.is_file():
            return 候选
    命中 = next(cwd.rglob(文件名), None)  # 子目录兜底（课程目录文件不多，开销可忽略）
    if 命中 is not None:
        return 命中
    raise FileNotFoundError(f"在 {cwd} 及其上、下级目录中都找不到 {文件名}，请把内核工作目录切到 notebook 所在文件夹。")

周报文件路径 = 定位周报文件()
print(f"读取数据：{周报文件路径}")  # 明确告诉学生到底读的是哪一个文件

奶茶店周报数据 = pd.read_csv(周报文件路径, encoding="utf-8-sig")  # 中文表头：utf-8-sig 兼容带 BOM 的 Excel 导出文件

读取数据：d:\桌面\silent-madman\Python基础语法课程\ttttt\奶茶店周报.csv


In [11]:
if "奶茶店周报数据" not in globals():  # 单独运行本单元格时给出明确提示，避免 NameError
    raise RuntimeError("请先运行上面的单元格（其中读取了 奶茶店周报数据）。")

# 布尔索引：先算出一列 True/False，再用它筛行
# 注意：奶茶 是销量（整数），写 == True 等价于“只挑销量等于 1 的行”，结果永远是空表
# 阈值取 150（销量区间是 110~260），才能真正看到“有些行被筛掉”；换成 > 100 会一行不落全选中
筛选条件 = 奶茶店周报数据["奶茶"] >= 150
result = 奶茶店周报数据[筛选条件]
result

,日期,奶茶,果茶,单价,总销量,营业额
4,周五,180,120,13,300,3900
5,周六,260,190,15,450,6750
6,周日,230,160,15,390,5850


In [12]:
if "result" not in globals():  # 单独运行本单元格时给出明确提示，避免 NameError
    raise RuntimeError("请先运行上面的单元格（其中筛选出了 result）。")

# 与数据文件同目录，避免相对路径把文件写到未知的工作目录
结果路径 = 周报文件路径.with_name("result.csv")
# index=False：不把行号写进去；encoding="utf-8-sig"：Excel/WPS 打开中文不乱码
result.to_csv(结果路径, index=False, encoding="utf-8-sig")
print(f"已保存：{结果路径}")

已保存：d:\桌面\silent-madman\Python基础语法课程\ttttt\result.csv


In [13]:
奶茶店周报数据.columns

Index(['日期', '奶茶', '果茶', '单价', '总销量', '营业额'], dtype='str')

In [14]:
# 列选择要用列表：集合无序（顺序不可控），pandas 更是直接禁止用 set 当索引器（TypeError）
# 另外这张表没有「咖啡」「果汁」两列，取不存在的列名会 KeyError
奶茶店周报数据.loc[3, ["奶茶", "果茶", "单价"]]

奶茶    142
果茶     88
单价     13
Name: 3, dtype: int64

In [15]:
def 奶茶店查询函数(df):
    需要的列 = {"奶茶", "果茶"}
    缺少的列 = 需要的列 - set(df.columns)  # 先校验列名，缺列时报错比 KeyError 更清楚
    if 缺少的列:
        raise KeyError(f"表格缺少列：{缺少的列}，现有列：{list(df.columns)}")
    条件1 = df["奶茶"] > 100
    条件2 = df["果茶"] > 100
    return df[条件1 & 条件2]

# 函数返回的已经是筛选后的 DataFrame，外面再套一层 .loc[...index] 是白跑一趟（多余的一次索引拷贝）
奶茶店查询函数(奶茶店周报数据).head(3)

,日期,奶茶,果茶,单价,总销量,营业额
4,周五,180,120,13,300,3900
5,周六,260,190,15,450,6750
6,周日,230,160,15,390,5850


In [16]:
奶茶店周报数据.query("奶茶 > 100 and 果茶 > 100").head(3)

,日期,奶茶,果茶,单价,总销量,营业额
4,周五,180,120,13,300,3900
5,周六,260,190,15,450,6750
6,周日,230,160,15,390,5850


In [17]:
奶茶店周报数据.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   日期      7 non-null      str  
 1   奶茶      7 non-null      int64
 2   果茶      7 non-null      int64
 3   单价      7 non-null      int64
 4   总销量     7 non-null      int64
 5   营业额     7 non-null      int64
dtypes: int64(5), str(1)
memory usage: 468.0 bytes


In [18]:
# describe() 本身是 8 行统计摘要，再 .count() 数的是摘要行数（恒为 8），不是样本数
print(奶茶店周报数据["奶茶"].describe())  # 统计摘要（用 [] 取值，属性写法遇到空格/关键字列名会失效）
奶茶店周报数据["奶茶"].count()  # 这才是非空样本数

count      7.000000
mean     168.142857
std       57.568593
min      110.000000
25%      127.500000
50%      142.000000
75%      205.000000
max      260.000000
Name: 奶茶, dtype: float64


np.int64(7)